# `klygo.files`: Paths

Construct, inspect, and map paths without changing files. A `Path` result is still only a value until another operation creates or modifies an entry.

Run this notebook from top to bottom. It creates only temporary example files.

In [ ]:
!pip install git+https://github.com/IchigoMazone/klygo.git@v2-release

## Prepare sample files

Run this setup again to reset the examples. No Drive mount or manual upload is required.

In [ ]:
from tempfile import TemporaryDirectory
from klygo import files

# Run this cell again to replace only this notebook's temporary workspace.
if "tutorial_workspace" in globals():
    tutorial_workspace.cleanup()
tutorial_workspace = TemporaryDirectory(prefix="klygo-files-")
root = files.path(tutorial_workspace.name)
print("Example directory:", root)

In [2]:
(root / "images").mkdir()
(root / "images" / "sample.jpg").write_bytes(b"sample image bytes")
image_path = root / "images" / "sample.jpg"
print("Sample path:", image_path)

Sample path: /tmp/klygo-files-tutorial-qopagp1a/images/sample.jpg


## Jump to an API

[`path`](#path) · [`join`](#join) · [`normalize`](#normalize) · [`resolve`](#resolve) · [`relative`](#relative) · [`as_posix`](#as_posix) · [`name`](#name) · [`stem`](#stem) · [`extension`](#extension) · [`extensions`](#extensions) · [`compound_extension`](#compound_extension) · [`parent`](#parent) · [`parents`](#parents) · [`is_absolute`](#is_absolute) · [`is_within`](#is_within) · [`common_path`](#common_path) · [`replace_root`](#replace_root) · [`with_name`](#with_name) · [`with_stem`](#with_stem) · [`with_extension`](#with_extension) · [`unique_path`](#unique_path)

<a id="path"></a>

### `files.path`

Convert a string or path object to `pathlib.Path`.

**Signature:** `files.path(value: 'PathInput', expand_user: 'bool' = True) -> 'Path'`

- `value`: Path-like value to transform.
- `expand_user`: Expand a leading ``~`` to the current user's home.

**Returns:** `pathlib.Path` — Native path object; expands home markers but does not resolve ``..``.

In [3]:
print(files.path('images/sample.jpg'))
print(type(files.path('images/sample.jpg')).__name__)

images/sample.jpg
PosixPath


**Another useful pattern**

In [ ]:
home_literal = files.path('~/datasets', expand_user=False)
home_expanded = files.path('~/datasets')
print(home_literal, home_expanded)
print(files.path(root / 'images'))

**Expected error and recovery**

In [ ]:
try:
    files.path(123)
except TypeError as error:
    print("TypeError:", error)

# Fix: pass a string or another os.PathLike[str] value.
print(files.path("123"))

<a id="join"></a>

### `files.join`

Join one or more path components.

**Signature:** `files.join(*parts: 'PathInput') -> 'Path'`

- `parts`: One or more path components.

**Returns:** `pathlib.Path` — Joined path.

A later absolute component can reset earlier components. This constructs a path; it does not copy or move any file.

In [4]:
image_path = files.join(root, 'images', 'sample.jpg')
print(image_path)

/tmp/klygo-files-tutorial-qopagp1a/images/sample.jpg


**Another useful pattern**

In [ ]:
dataset_root = files.join(root, 'dataset')
planned_image = files.join(dataset_root, 'images', 'cat.jpg')
print(planned_image, files.exists(planned_image))

**Expected error and recovery**

In [ ]:
try:
    files.join()
except ValueError as error:
    print("ValueError:", error)

print(files.join(root, "images"))

<a id="normalize"></a>

### `files.normalize`

Normalize separators and dot components in a path.

**Signature:** `files.normalize(value: 'PathInput') -> 'Path'`

- `value`: Path-like value to transform.

**Returns:** `pathlib.Path` — Lexically normalized path.

This is lexical normalization. It collapses `.` and `..` but does not inspect symlink targets; use `resolve()` for a filesystem-aware absolute path.

In [5]:
print(files.normalize('images/./draft/../sample.jpg'))

images/sample.jpg


**Another useful pattern**

In [ ]:
messy = files.join(root, 'images', '..', 'labels', 'cat.txt')
print('Before:', messy)
print('After:', files.normalize(messy))

<a id="resolve"></a>

### `files.resolve`

Return an absolute normalized path.

**Signature:** `files.resolve(value: 'PathInput', strict: 'bool' = False) -> 'Path'`

- `value`: Path-like value to transform.
- `strict`: Require the resolved path to exist.

**Returns:** `pathlib.Path` — Absolute normalized path.

In [6]:
print(files.resolve(image_path, strict=True))
try:
    files.resolve(root / 'missing.jpg', strict=True)
except FileNotFoundError as error:
    print(type(error).__name__, '- use strict=False for a prospective path')

/tmp/klygo-files-tutorial-qopagp1a/images/sample.jpg
FileNotFoundError - use strict=False for a prospective path


**Another useful pattern**

In [ ]:
planned = files.resolve(root / 'future' / 'report.json')
print('Absolute path for a file not created yet:', planned)
print('Existing image:', files.resolve(image_path, strict=True))

**Expected error and recovery**

In [ ]:
try:
    files.resolve(root / "missing.txt", strict=True)
except FileNotFoundError as error:
    print("FileNotFoundError:", error)

print(files.resolve(root / "missing.txt", strict=False))

<a id="relative"></a>

### `files.relative`

Compute a path relative to a starting directory.

**Signature:** `files.relative(value: 'PathInput', start: 'PathInput' = '.') -> 'Path'`

- `value`: Path-like value to transform.
- `start`: Starting directory used to compute the relative path.

**Returns:** `pathlib.Path` — Path relative to ``start``.

This uses `os.path.relpath`, so the result can begin with `..`. It does not require the target to be inside `start`.

In [7]:
print(files.relative(image_path, root))

images/sample.jpg


**Another useful pattern**

In [ ]:
for item in (image_path, root / 'labels' / 'sample.txt'):
    print(files.relative(item, root))

<a id="as_posix"></a>

### `files.as_posix`

Return a path string with forward-slash separators.

**Signature:** `files.as_posix(value: 'PathInput') -> 'str'`

- `value`: Path to represent.

**Returns:** `str` — Path in POSIX separator form without resolving it.

In [8]:
print(files.as_posix(files.join('images', 'sample.jpg')))

images/sample.jpg


**Another useful pattern**

In [ ]:
manifest_entry = files.as_posix(files.relative(image_path, root))
print('Portable manifest path:', manifest_entry)

<a id="name"></a>

### `files.name`

Return the final path component including its extension.

**Signature:** `files.name(value: 'PathInput') -> 'str'`

- `value`: Path to inspect or operate on.

**Returns:** `str` — Final path component.

In [9]:
print(files.name('archive.tar.gz'))

archive.tar.gz


**Another useful pattern**

In [ ]:
for item in ('cat.jpg', 'archive.tar.gz', '.env'):
    print(item, '->', files.name(item))

<a id="stem"></a>

### `files.stem`

Return the final path component without its last extension.

**Signature:** `files.stem(value: 'PathInput') -> 'str'`

- `value`: Path to inspect or operate on.

**Returns:** `str` — Final component without its last suffix.

Only the final suffix is removed: `archive.tar.gz` has stem `archive.tar`.

In [10]:
print(files.stem('archive.tar.gz'))

archive.tar


**Another useful pattern**

In [ ]:
for item in ('cat.jpg', 'archive.tar.gz'):
    print(item, '->', files.stem(item))

<a id="extension"></a>

### `files.extension`

Return the final filename extension.

**Signature:** `files.extension(value: 'PathInput') -> 'str'`

- `value`: Path to inspect or operate on.

**Returns:** `str` — Final suffix, including the leading dot.

Only the final suffix is returned. Use `extensions()` or `compound_extension()` when the full suffix chain matters.

In [11]:
print(files.extension('archive.tar.gz'))

.gz


**Another useful pattern**

In [ ]:
for item in ('cat.jpg', 'archive.tar.gz', 'README'):
    print(item, '->', repr(files.extension(item)))

<a id="extensions"></a>

### `files.extensions`

Return every filename suffix as a tuple.

**Signature:** `files.extensions(value: 'PathInput') -> 'Tuple[str, ...]'`

- `value`: Path-like value to transform.

**Returns:** `tuple[str, ...]` — All suffixes in filename order.

In [12]:
print(files.extensions('archive.tar.gz'))

('.tar', '.gz')


**Another useful pattern**

In [ ]:
for item in ('archive.tar.gz', 'photo.jpg'):
    print(item, '->', files.extensions(item))

<a id="compound_extension"></a>

### `files.compound_extension`

Return every filename suffix joined together.

**Signature:** `files.compound_extension(value: 'PathInput') -> 'str'`

- `value`: Path-like value to transform.

**Returns:** `str` — All suffixes joined together.

All suffixes are joined, not just the archive suffixes recognized by `with_extension()`.

In [13]:
print(files.compound_extension('archive.tar.gz'))

.tar.gz


**Another useful pattern**

In [ ]:
for item in ('archive.tar.gz', 'backup.sql.gz', 'photo.jpg'):
    print(item, '->', files.compound_extension(item))

<a id="parent"></a>

### `files.parent`

Return the immediate parent path.

**Signature:** `files.parent(value: 'PathInput') -> 'Path'`

- `value`: Path to inspect or operate on.

**Returns:** `pathlib.Path` — Immediate parent path.

In [14]:
print(files.parent(image_path))

/tmp/klygo-files-tutorial-qopagp1a/images


**Another useful pattern**

In [ ]:
print('Image directory:', files.parent(image_path))
print('Dataset root:', files.parent(files.parent(image_path)))

<a id="parents"></a>

### `files.parents`

Return every ancestor path from nearest to farthest.

**Signature:** `files.parents(value: 'PathInput') -> 'Tuple[Path, ...]'`

- `value`: Path-like value to transform.

**Returns:** `tuple[pathlib.Path, ...]` — Ancestors from nearest to farthest.

In [15]:
print(files.parents(image_path)[:2])

(PosixPath('/tmp/klygo-files-tutorial-qopagp1a/images'), PosixPath('/tmp/klygo-files-tutorial-qopagp1a'))


**Another useful pattern**

In [ ]:
for ancestor in files.parents(image_path)[:3]:
    print(ancestor)

<a id="is_absolute"></a>

### `files.is_absolute`

Check whether a path is absolute.

**Signature:** `files.is_absolute(value: 'PathInput') -> 'bool'`

- `value`: Path-like value to transform.

**Returns:** `bool` — Whether the value is absolute.

In [16]:
print(files.is_absolute(image_path))
print(files.is_absolute('images/sample.jpg'))

True
False


**Another useful pattern**

In [ ]:
candidate = files.join(root, 'output', 'result.json')
print(files.is_absolute(candidate))
print(files.is_absolute('output/result.json'))

<a id="is_within"></a>

### `files.is_within`

Check whether a path is contained by a root directory.

**Signature:** `files.is_within(value: 'PathInput', root: 'PathInput', resolve_paths: 'bool' = True) -> 'bool'`

- `value`: Path-like value to transform.
- `root`: Root directory for the containment check.
- `resolve_paths`: Resolve both paths before testing containment.

**Returns:** `bool` — Whether ``value`` is contained by ``root``.

With the default `resolve_paths=True`, symlinks and `..` are resolved before containment is checked. A path counts as within itself.

In [17]:
print(files.is_within(image_path, root))
print(files.is_within(root.parent / 'other.jpg', root))

True
False


**Another useful pattern**

In [ ]:
safe = files.is_within(root / 'images' / 'cat.jpg', root)
outside = files.is_within(root / '..' / 'outside.jpg', root)
print('Inside:', safe, 'outside:', outside)

<a id="common_path"></a>

### `files.common_path`

Return the longest common parent of multiple paths.

**Signature:** `files.common_path(values: 'Iterable[PathInput]') -> 'Path'`

- `values`: Non-empty collection of paths.

**Returns:** `pathlib.Path` — Longest common parent.

Supply a non-empty iterable. The calculation is lexical; on Windows, paths on different drives can raise `ValueError`.

In [18]:
print(files.common_path([root / 'images' / 'a.jpg', root / 'images' / 'b.jpg']))

/tmp/klygo-files-tutorial-qopagp1a/images


**Another useful pattern**

In [ ]:
dataset_paths = [root / 'images' / 'cat.jpg', root / 'images' / 'dog.jpg']
print('Shared directory:', files.common_path(dataset_paths))

**Expected error and recovery**

In [ ]:
try:
    files.common_path([])
except ValueError as error:
    print("ValueError:", error)

print(files.common_path([root / "images", root / "labels"]))

<a id="replace_root"></a>

### `files.replace_root`

Map a path from one directory tree into another.

**Signature:** `files.replace_root(value: 'PathInput', old_root: 'PathInput', new_root: 'PathInput') -> 'Path'`

- `value`: Path-like value to transform.
- `old_root`: Root removed from the input path.
- `new_root`: Replacement root.

**Returns:** `pathlib.Path` — Mapped path below ``new_root``.

This maps the path's relative location; it does not move data. A value outside `old_root` raises `ValueError`.

In [19]:
print(files.replace_root(image_path, root / 'images', root / 'backup'))

/tmp/klygo-files-tutorial-qopagp1a/backup/sample.jpg


**Another useful pattern**

In [ ]:
label_path = files.replace_root(image_path, root / 'images', root / 'labels')
label_path = files.with_extension(label_path, '.txt')
print('Matching label path:', label_path)

**Expected error and recovery**

In [ ]:
try:
    files.replace_root(root.parent / "outside.jpg", root, root / "backup")
except ValueError as error:
    print("ValueError:", error)

<a id="with_name"></a>

### `files.with_name`

Return a path with a replaced final component.

**Signature:** `files.with_name(value: 'PathInput', new_name: 'str') -> 'Path'`

- `value`: Path-like value to transform.
- `new_name`: Replacement final path component.

**Returns:** `pathlib.Path` — Path with the new final component.

In [20]:
print(files.with_name(image_path, 'renamed.jpg'))

/tmp/klygo-files-tutorial-qopagp1a/images/renamed.jpg


**Another useful pattern**

In [ ]:
print(files.with_name(image_path, 'dog.jpg'))
print('Original image stays:', files.exists(image_path))

**Expected error and recovery**

In [ ]:
try:
    files.with_name(image_path, "nested/name.jpg")
except ValueError as error:
    print("ValueError:", error)

print(files.with_name(image_path, "valid-name.jpg"))

<a id="with_stem"></a>

### `files.with_stem`

Return a path with a replaced stem.

**Signature:** `files.with_stem(value: 'PathInput', new_stem: 'str') -> 'Path'`

- `value`: Path-like value to transform.
- `new_stem`: Replacement filename stem.

**Returns:** `pathlib.Path` — Path with the new stem.

In [21]:
print(files.with_stem(image_path, 'renamed'))

/tmp/klygo-files-tutorial-qopagp1a/images/renamed.jpg


**Another useful pattern**

In [ ]:
renamed_candidate = files.with_stem(image_path, 'dog')
print(renamed_candidate, files.extension(renamed_candidate))

<a id="with_extension"></a>

### `files.with_extension`

Return a path with a replaced filename extension.

**Signature:** `files.with_extension(value: 'PathInput', new_extension: 'str') -> 'Path'`

- `value`: Path-like value to transform.
- `new_extension`: Replacement extension, with or without a leading dot.

**Returns:** `pathlib.Path` — Path with the new extension.

Known archive suffixes `.tar.gz`, `.tar.xz`, and `.tar.bz2` are replaced together. For other multi-suffix names, only the final suffix is replaced.

In [22]:
print(files.with_extension('archive.tar.gz', '.zip'))
print(files.with_extension(image_path, 'png'))

archive.zip
/tmp/klygo-files-tutorial-qopagp1a/images/sample.png


**Another useful pattern**

In [ ]:
for item in ('archive.tar.gz', 'sample.jpg', 'notes'):
    print(item, '->', files.with_extension(item, 'zip'))

**Expected error and recovery**

In [ ]:
try:
    files.with_extension(image_path, "folder/png")
except ValueError as error:
    print("ValueError:", error)

print(files.with_extension(image_path, "png"))

<a id="unique_path"></a>

### `files.unique_path`

Return a currently unused path.

**Signature:** `files.unique_path(value: 'PathInput', separator: 'str' = '_', start: 'int' = 1) -> 'Path'`

- `value`: Path-like value to transform.
- `separator`: Text inserted before the numeric suffix.
- `start`: Initial numeric suffix.

**Returns:** `pathlib.Path` — An unused candidate path.

The result is only an unused candidate at that instant. Another process may claim it before you write; `save()` or `copy()` with `overwrite=False` enforces the final no-overwrite check.

In [23]:
print(files.unique_path(image_path))
print(files.unique_path(image_path, separator='-', start=10))

/tmp/klygo-files-tutorial-qopagp1a/images/sample_1.jpg
/tmp/klygo-files-tutorial-qopagp1a/images/sample-10.jpg


**Another useful pattern**

In [ ]:
for separator in ('_', '-'):
    print(files.unique_path(image_path, separator=separator, start=2))

**Expected error and recovery**

In [ ]:
try:
    files.unique_path(image_path, start=-1)
except ValueError as error:
    print("ValueError:", error)

## Put it together

In [24]:
destination = files.replace_root(image_path, root / 'images', root / 'archive')
print('Planned destination:', files.with_extension(destination, '.png'))
print('Source still exists:', files.exists(image_path))

Planned destination: /tmp/klygo-files-tutorial-qopagp1a/archive/sample.png
Source still exists: True


## Remember

A path helper returns a value; it does not rename, copy, or create the file. For those operations, continue with [filesystem.ipynb](filesystem.ipynb).

The examples above use this temporary directory only. See [the overview](overview.ipynb) for the full learning path.